In [76]:
import pandas as pd
import requests
import json
import re
from io import StringIO

url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

fixtures = []

for table in tables:
    columns = list(table.columns)

    # We only want tables that look like:
    # Team A | Match X | Team B
    if len(columns) == 3 and "Match" in str(columns[1]):
        home_team = str(columns[0]).strip()
        match_text = str(columns[1]).strip()
        away_team = str(columns[2]).strip()

        match_number = re.search(r"\d+", match_text)

        if not match_number:
            continue

        fixtures.append({
            "matchId": int(match_number.group()) if match_number else None,
            "homeTeam": home_team,
            "awayTeam": away_team,
            "homeTeamKey": home_team.lower().replace(" ", "_"),
            "awayTeamKey": away_team.lower().replace(" ", "_"),
            "matchImportance": 5,
            "rivalryScore": 5
        })

fixtures = sorted(fixtures, key=lambda x: x["matchId"])

with open("fixtures.json", "w") as f:
    json.dump({"fixtures": fixtures}, f, indent=2)

print(f"Saved {len(fixtures)} fixtures to fixtures.json")


Saved 102 fixtures to fixtures.json


In [ ]:
import json
from pathlib import Path

BASE_DIR = Path("data")

with open(BASE_DIR / "teams.json", "r") as f:
    teams = json.load(f)

with open(BASE_DIR / "fixtures.json", "r") as f:
    fixtures_data = json.load(f)

fixtures = fixtures_data["fixtures"]

def normalize(value, min_value, max_value):
    score = ((value - min_value) / (max_value - min_value)) * 10
    return max(0, min(score, 10))


def calculate_excitement(home_key, away_key, match_importance, rivalry_score):
    home_team = teams[home_key]
    away_team = teams[away_key]

    avg_goals_for = (
        home_team["goalsForPerMatch"] + away_team["goalsForPerMatch"]
    ) / 2

    avg_goals_against = (
        home_team["goalsAgainstPerMatch"] + away_team["goalsAgainstPerMatch"]
    ) / 2

    avg_star_power = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    elo_difference = abs(home_team["eloRating"] - away_team["eloRating"])

    attacking_score = normalize(avg_goals_for, 0, 3)
    chaos_score = normalize(avg_goals_against, 0, 3)
    # Smaller Elo gap = more balanced game = higher score
    competitive_balance = 10 - normalize(elo_difference, 0, 600)

    excitement_score = (
        0.25 * attacking_score +
        0.20 * chaos_score +
        0.20 * avg_star_power +
        0.15 * competitive_balance +
        0.10 * match_importance +
        0.10 * rivalry_score) * 10

    team_quality_score = (
        home_team["starPowerScore"] + away_team["starPowerScore"]
    ) / 2

    quality_floor = team_quality_score * 7.5

    excitement_score = max(excitement_score, quality_floor)

    return {
        "finalScore": round(excitement_score, 1),
        "breakdown": {
            "attackingScore": round(attacking_score, 2),
            "chaosScore": round(chaos_score, 2),
            "starPowerScore": round(avg_star_power, 2),
            "competitiveBalance": round(competitive_balance, 2),
            "matchImportance": match_importance,
            "rivalryScore": rivalry_score
        }
    }

def get_watch_tier(score):
        if score >= 75:
            return "Match of the tournament candidate"
        elif score >= 65:
            return "Must watch"
        elif score >= 60:
            return "Strong watch"
        elif score >=50:
            return "Background watch"
        else:
            return "Skippable unless you support them"

scored_fixtures = []

for fixture in fixtures:
    home_key = fixture["homeTeamKey"]
    away_key = fixture["awayTeamKey"]

    if home_key not in teams:
        print(f"Skipping: {home_key} not found")
        continue

    if away_key not in teams:
        print(f"Skipping: {away_key} not found")
        continue

    result = calculate_excitement(
        home_key,
        away_key,
        fixture.get("matchImportance", 5),
        fixture.get("rivalryScore", 5)
    )

    watch_tier = get_watch_tier(result["finalScore"])

    scored_fixture = {
        **fixture,
        "homeTeamName": teams[home_key]["teamName"],
        "awayTeamName": teams[away_key]["teamName"],
        "homeElo": teams[home_key]["eloRating"],
        "awayElo": teams[away_key]["eloRating"],
        "homeStarPower": teams[home_key]["starPowerScore"],
        "awayStarPower": teams[away_key]["starPowerScore"],
        "starPlayers": (
            teams[home_key].get("starPlayers", []) +
            teams[away_key].get("starPlayers", [])
        ),
        "whyWatch": fixture.get("whyWatch", ""),
        "excitementScore": result["finalScore"],
        "watchTier": watch_tier,
        "scoreBreakdown": result["breakdown"],
    }

    scored_fixtures.append(scored_fixture)


scored_fixtures = sorted(
    scored_fixtures,
    key=lambda x:x["excitementScore"],
    reverse=True
)

with open(BASE_DIR / "scored_fixtures.json", "w") as f:
    json.dump({"fixtures": scored_fixtures}, f, indent=2)

for fixture in scored_fixtures:
    print(
        f"{fixture['homeTeamName']} vs {fixture['awayTeamName']}: "
        f"{fixture['excitementScore']} - {fixture['watchTier']}"
    )

Skipping: runner-up_group_a not found
Skipping: winner_group_e not found
Skipping: winner_group_f not found
Skipping: winner_group_c not found
Skipping: winner_group_i not found
Skipping: runner-up_group_e not found
Skipping: winner_group_a not found
Skipping: winner_group_l not found
Skipping: winner_group_d not found
Skipping: winner_group_g not found
Skipping: runner-up_group_k not found
Skipping: winner_group_h not found
Skipping: winner_group_b not found
Skipping: winner_group_j not found
Skipping: winner_group_k not found
Skipping: runner-up_group_d not found
Skipping: winner_match_74 not found
Skipping: winner_match_73 not found
Skipping: winner_match_76 not found
Skipping: winner_match_79 not found
Skipping: winner_match_83 not found
Skipping: winner_match_81 not found
Skipping: winner_match_86 not found
Skipping: winner_match_85 not found
Skipping: winner_match_89 not found
Skipping: winner_match_93 not found
Skipping: winner_match_91 not found
Skipping: winner_match_95 not fo